# PaperBrief — fast paper triage in Jupyter

Paste a paper or upload a PDF, pick an [OpenRouter](https://openrouter.ai) model, and get a
skim‑ready brief: **TL;DR**, method tags, and five sections —
**Contribution / Identification / Data / Results / Caveats**. Everything is saved to a local
SQLite dashboard you can search and revisit.

**How to use**
1. Run the *Setup* cells (install + config). Set your key via the `OPENROUTER_API_KEY`
   environment variable, or you'll be prompted (hidden input). The key is never stored in this notebook.
2. Run the *Interactive app* cell, paste/upload a paper, choose a model, click **Analyze**, then **Save**.
3. Run the *Dashboard* cell to browse, view, and delete saved briefs.

> Security: your key was shared in chat earlier — rotate it at openrouter.ai/keys and paste the new one when prompted.

In [ ]:
# Setup 1/5 — install dependencies (safe to re-run)
%pip install -q requests ipywidgets pypdf pandas

In [ ]:
# Setup 2/5 — configuration (API key, default model, database path)
import os, io, json, time, uuid, sqlite3
from datetime import datetime
import requests

# Your OpenRouter API key is read from the environment. If it is not set, you are
# prompted with a hidden input. It is never written into this notebook or the database.
OPENROUTER_API_KEY = os.environ.get("OPENROUTER_API_KEY", "").strip()
if not OPENROUTER_API_KEY:
    try:
        from getpass import getpass
        OPENROUTER_API_KEY = getpass("Enter your OpenRouter API key (hidden): ").strip()
    except Exception:
        print("No API key found. Set OPENROUTER_API_KEY or re-run this cell to be prompted.")

DEFAULT_MODEL = "openai/gpt-4o-mini"          # change to any OpenRouter model id
DB_PATH = os.path.join("data", "paperbrief.db")
os.makedirs("data", exist_ok=True)

print(f"API key set: {bool(OPENROUTER_API_KEY)}  |  default model: {DEFAULT_MODEL}  |  db: {DB_PATH}")

In [ ]:
# Setup 3/5 — OpenRouter client (list models + analyze a paper)
OPENROUTER_BASE = "https://openrouter.ai/api/v1"
MAX_INPUT_CHARS = 120_000  # bound cost/context; longer inputs are truncated

SYSTEM_PROMPT = """You are an expert research economist who triages empirical and theoretical papers for a busy professor. You produce fast, skimmable briefs.

You will be given the text of a paper (it may be truncated). Extract a structured brief and respond with ONLY a JSON object (no markdown, no prose) matching exactly this shape:

{
  "title": string,
  "authors": string[],
  "year": string,
  "tldr": string,               // one crisp sentence: the paper's punchline
  "methodTags": string[],       // short method labels, e.g. "RCT","DiD","IV","RDD","Event study","Structural","Panel FE","Theory","Meta-analysis"
  "contribution": string[],     // what is new and why it matters
  "identification": string[],   // research design, identifying variation, key assumptions, threats to identification
  "data": string[],             // datasets/sources, sample size, time period, unit of observation, key variables (treatment & outcome)
  "results": string[],          // main estimates with MAGNITUDES, signs, significance, and notable heterogeneity
  "caveats": string[]           // limitations, external validity, robustness, potential confounds
}

Rules:
- Every array item is a short, self-contained bullet (ideally under 25 words). Prefer specific numbers over vague statements.
- Be faithful to the paper; do not invent results. If something is not reported, use a single bullet "Not reported" for that section.
- For a purely theoretical paper, use "identification" to describe the modeling framework and key assumptions.
- Output valid minified JSON only."""


def list_models(api_key=None):
    """Return available OpenRouter models as [{'id','name','context'}, ...]."""
    key = api_key or OPENROUTER_API_KEY
    headers = {"Authorization": f"Bearer {key}"} if key else {}
    r = requests.get(f"{OPENROUTER_BASE}/models", headers=headers, timeout=60)
    r.raise_for_status()
    data = r.json().get("data", [])
    models = [
        {"id": m["id"], "name": m.get("name", m["id"]), "context": m.get("context_length")}
        for m in data
    ]
    return sorted(models, key=lambda m: m["name"].lower())


def _to_list(value):
    if isinstance(value, list):
        return [str(x).strip() for x in value if str(x).strip()]
    if isinstance(value, str) and value.strip():
        return [value.strip()]
    return []


def normalize_brief(raw):
    """Coerce arbitrary model JSON into a well-formed brief dict."""
    raw = raw or {}
    return {
        "title": str(raw.get("title") or "Untitled paper").strip(),
        "authors": _to_list(raw.get("authors")),
        "year": str(raw.get("year") or "").strip(),
        "tldr": str(raw.get("tldr") or "").strip(),
        "methodTags": _to_list(raw.get("methodTags") or raw.get("method_tags")),
        "contribution": _to_list(raw.get("contribution")),
        "identification": _to_list(raw.get("identification") or raw.get("id")),
        "data": _to_list(raw.get("data")),
        "results": _to_list(raw.get("results")),
        "caveats": _to_list(raw.get("caveats")),
    }


def _extract_json(content):
    content = content.strip()
    try:
        return json.loads(content)
    except Exception:
        start, end = content.find("{"), content.rfind("}")
        if start != -1 and end != -1 and end > start:
            return json.loads(content[start : end + 1])
        raise ValueError("Model did not return valid JSON.")


def analyze_paper(text, model=None, api_key=None):
    """Send paper text to OpenRouter and return {'brief','usage','truncated'}."""
    key = api_key or OPENROUTER_API_KEY
    if not key:
        raise ValueError("Missing OpenRouter API key. Set OPENROUTER_API_KEY in Setup 2.")
    model = model or DEFAULT_MODEL
    truncated = len(text) > MAX_INPUT_CHARS
    payload_text = text[:MAX_INPUT_CHARS] if truncated else text

    resp = requests.post(
        f"{OPENROUTER_BASE}/chat/completions",
        headers={
            "Authorization": f"Bearer {key}",
            "Content-Type": "application/json",
            "X-Title": "PaperBrief (Jupyter)",
        },
        json={
            "model": model,
            "temperature": 0.2,
            "response_format": {"type": "json_object"},
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {
                    "role": "user",
                    "content": "Analyze the following paper and return the JSON brief."
                    + (" (NOTE: text was truncated to fit context.)" if truncated else "")
                    + f"\n\n===== PAPER TEXT =====\n{payload_text}",
                },
            ],
        },
        timeout=180,
    )
    if not resp.ok:
        raise RuntimeError(f"OpenRouter request failed ({resp.status_code}): {resp.text[:400]}")
    data = resp.json()
    if data.get("error"):
        raise RuntimeError(data["error"].get("message", "OpenRouter error"))
    content = data["choices"][0]["message"]["content"]
    return {
        "brief": normalize_brief(_extract_json(content)),
        "usage": data.get("usage"),
        "truncated": truncated,
    }

print("OpenRouter client ready: list_models(), analyze_paper()")

In [ ]:
# Setup 4/5 — SQLite storage (original text + structured brief)
def _connect():
    con = sqlite3.connect(DB_PATH)
    con.row_factory = sqlite3.Row
    return con


def init_db():
    with _connect() as con:
        con.execute(
            """CREATE TABLE IF NOT EXISTS papers (
                id TEXT PRIMARY KEY,
                title TEXT NOT NULL,
                authors TEXT NOT NULL DEFAULT '[]',
                year TEXT NOT NULL DEFAULT '',
                model TEXT NOT NULL DEFAULT '',
                created_at INTEGER NOT NULL,
                brief TEXT NOT NULL,
                source_text TEXT NOT NULL DEFAULT ''
            )"""
        )


def save_brief(brief, model, source_text):
    rec_id, created = str(uuid.uuid4()), int(time.time() * 1000)
    with _connect() as con:
        con.execute(
            "INSERT INTO papers VALUES (?,?,?,?,?,?,?,?)",
            (
                rec_id,
                brief["title"],
                json.dumps(brief["authors"]),
                brief["year"],
                model,
                created,
                json.dumps(brief),
                source_text,
            ),
        )
    return rec_id


def list_briefs():
    with _connect() as con:
        rows = con.execute("SELECT * FROM papers ORDER BY created_at DESC").fetchall()
    result = []
    for r in rows:
        b = json.loads(r["brief"])
        result.append(
            {
                "id": r["id"],
                "title": r["title"],
                "authors": json.loads(r["authors"]),
                "year": r["year"],
                "model": r["model"],
                "created_at": r["created_at"],
                "tldr": b.get("tldr", ""),
                "methodTags": b.get("methodTags", []),
            }
        )
    return result


def get_brief(rec_id):
    with _connect() as con:
        r = con.execute("SELECT * FROM papers WHERE id = ?", (rec_id,)).fetchone()
    if not r:
        return None
    return {
        "id": r["id"],
        "model": r["model"],
        "created_at": r["created_at"],
        "brief": json.loads(r["brief"]),
        "source_text": r["source_text"],
    }


def delete_brief(rec_id):
    with _connect() as con:
        cur = con.execute("DELETE FROM papers WHERE id = ?", (rec_id,))
    return cur.rowcount > 0


init_db()
print(f"Storage ready — {len(list_briefs())} brief(s) saved so far.")

In [ ]:
# Setup 5/5 — file extraction + pretty display
import html as _html
from IPython.display import HTML, display

SECTIONS = [
    ("contribution", "Contribution", "#6366f1"),
    ("identification", "Identification", "#10b981"),
    ("data", "Data", "#f59e0b"),
    ("results", "Results", "#0ea5e9"),
    ("caveats", "Caveats", "#f43f5e"),
]


def extract_text(file_bytes, filename=""):
    """Extract text from PDF bytes (via pypdf) or decode text files."""
    is_pdf = filename.lower().endswith(".pdf") or file_bytes[:5] == b"%PDF-"
    if is_pdf:
        from pypdf import PdfReader
        reader = PdfReader(io.BytesIO(file_bytes))
        return "\n".join((page.extract_text() or "") for page in reader.pages).strip()
    return file_bytes.decode("utf-8", errors="ignore").strip()


def brief_html(brief):
    def esc(s):
        return _html.escape(str(s))

    tags = "".join(
        f'<span style="background:#f1f5f9;border-radius:999px;padding:2px 10px;'
        f'margin-right:6px;font-size:12px;color:#334155">{esc(t)}</span>'
        for t in brief["methodTags"]
    )
    parts = [
        f'<div style="font-family:system-ui,-apple-system,Segoe UI,Roboto,sans-serif;'
        f'max-width:760px;line-height:1.5">',
        f'<h2 style="margin:0 0 2px">{esc(brief["title"])}</h2>',
        f'<div style="color:#64748b;font-size:13px;margin-bottom:8px">'
        f'{esc(", ".join(brief["authors"]))}{" · " + esc(brief["year"]) if brief["year"] else ""}</div>',
    ]
    if brief["tldr"]:
        parts.append(
            f'<div style="background:#eef2ff;color:#3730a3;border-radius:8px;'
            f'padding:8px 12px;margin:8px 0"><b>TL;DR</b> {esc(brief["tldr"])}</div>'
        )
    if tags:
        parts.append(f'<div style="margin:8px 0">{tags}</div>')
    for key, label, color in SECTIONS:
        items = brief.get(key) or ["Not reported."]
        bullets = "".join(f"<li>{esc(i)}</li>" for i in items)
        parts.append(
            f'<div style="border-left:4px solid {color};background:#fff;'
            f'border:1px solid #e2e8f0;border-left:4px solid {color};'
            f'border-radius:8px;padding:10px 14px;margin:10px 0">'
            f'<div style="font-weight:600;margin-bottom:4px">{label}</div>'
            f'<ul style="margin:0;padding-left:18px;color:#334155">{bullets}</ul></div>'
        )
    parts.append("</div>")
    return "".join(parts)


def show_brief(brief):
    display(HTML(brief_html(brief)))

print("Helpers ready: extract_text(), show_brief()")

## Interactive app

Paste paper text or upload a PDF/txt, pick a model (type to search), click **Analyze**, then **Save to dashboard**.

In [ ]:
# Interactive app
import ipywidgets as widgets
from IPython.display import display, clear_output

try:
    _model_options = [m["id"] for m in list_models()]
except Exception as e:
    print("Could not fetch model list, using default. (", e, ")")
    _model_options = [DEFAULT_MODEL]

_model = widgets.Combobox(
    value=DEFAULT_MODEL, options=_model_options, placeholder="type to search models",
    description="Model:", ensure_option=False, layout=widgets.Layout(width="480px"),
)
_text = widgets.Textarea(
    placeholder="Paste the paper text (abstract + body) here…",
    layout=widgets.Layout(width="100%", height="220px"),
)
_upload = widgets.FileUpload(accept=".pdf,.txt,.md,.tex", multiple=False, description="Upload PDF/txt")
_analyze = widgets.Button(description="Analyze", button_style="primary", icon="search")
_save = widgets.Button(description="Save to dashboard", button_style="success", icon="save", disabled=True)
_status = widgets.HTML()
_out = widgets.Output()
_state = {"brief": None, "source": ""}


def _read_upload():
    """Return (filename, bytes) for the uploaded file across ipywidgets versions."""
    val = _upload.value
    if not val:
        return None, None
    item = list(val.values())[0] if isinstance(val, dict) else val[0]
    name = item.get("name") or (list(val.keys())[0] if isinstance(val, dict) else "upload")
    return name, bytes(item["content"])


def _on_analyze(_):
    _save.disabled = True
    _state["brief"] = None
    with _out:
        clear_output()
    text = _text.value.strip()
    name, content = _read_upload()
    if content and len(text) < 40:
        _status.value = "Reading file…"
        text = extract_text(content, name or "")
        _text.value = text
    if len(text) < 40:
        _status.value = '<span style="color:#dc2626">Please paste or upload a paper first.</span>'
        return
    _status.value = f"Analyzing with <code>{_model.value}</code> …"
    try:
        res = analyze_paper(text, _model.value)
    except Exception as e:
        _status.value = f'<span style="color:#dc2626">Error: {_html.escape(str(e))}</span>'
        return
    _state["brief"], _state["source"] = res["brief"], text
    note = " (input truncated to fit context)" if res["truncated"] else ""
    cost = (res.get("usage") or {}).get("cost")
    _status.value = f'<span style="color:#059669">Done{note}.</span>' + (
        f' <span style="color:#64748b">~${cost:.4f}</span>' if isinstance(cost, (int, float)) else ""
    )
    with _out:
        clear_output()
        show_brief(res["brief"])
    _save.disabled = False


def _on_save(_):
    if not _state["brief"]:
        return
    rec_id = save_brief(_state["brief"], _model.value, _state["source"])
    _status.value = f'<span style="color:#059669">Saved ✓</span> <code>{rec_id[:8]}</code> — run the Dashboard cell to see it.'
    _save.disabled = True


_analyze.on_click(_on_analyze)
_save.on_click(_on_save)

display(
    widgets.VBox(
        [
            widgets.HBox([_upload, _model]),
            _text,
            widgets.HBox([_analyze, _save, _status]),
            _out,
        ]
    )
)

## Dashboard

An overview table of every saved brief, plus an interactive viewer to open the full brief, see the
original text, or delete an entry. Re-run this cell after saving to refresh.

In [ ]:
# Dashboard
import pandas as pd
import ipywidgets as widgets
from IPython.display import display, clear_output


def dashboard_df():
    """Return a pandas DataFrame overview of all saved briefs."""
    rows = list_briefs()
    return pd.DataFrame(
        [
            {
                "id": r["id"][:8],
                "title": r["title"],
                "tldr": r["tldr"],
                "methods": ", ".join(r["methodTags"]),
                "model": r["model"],
                "saved": datetime.fromtimestamp(r["created_at"] / 1000).strftime("%Y-%m-%d %H:%M"),
            }
            for r in rows
        ],
        columns=["id", "title", "tldr", "methods", "model", "saved"],
    )


def show_dashboard():
    briefs = list_briefs()
    if not briefs:
        print("No saved briefs yet — analyze one in the Interactive app cell above.")
        return

    display(dashboard_df())

    options = [
        (f"{b['title'][:70]}  ·  {datetime.fromtimestamp(b['created_at']/1000):%Y-%m-%d %H:%M}", b["id"])
        for b in briefs
    ]
    picker = widgets.Dropdown(options=options, description="Open:", layout=widgets.Layout(width="640px"))
    show_src = widgets.Checkbox(value=False, description="Show original text")
    delete = widgets.Button(description="Delete", button_style="danger", icon="trash")
    view = widgets.Output()

    def render(*_):
        with view:
            clear_output()
            rec = get_brief(picker.value)
            if not rec:
                print("(deleted)")
                return
            show_brief(rec["brief"])
            if show_src.value:
                display(HTML(
                    '<details open><summary style="cursor:pointer;color:#64748b">Original text</summary>'
                    f'<pre style="white-space:pre-wrap;font-size:12px;max-height:360px;overflow:auto;'
                    f'background:#f8fafc;padding:10px;border-radius:8px">{_html.escape(rec["source_text"])}</pre></details>'
                ))

    def on_delete(_):
        delete_brief(picker.value)
        with view:
            clear_output()
            print("Deleted. Re-run this cell to refresh the list.")

    picker.observe(render, names="value")
    show_src.observe(render, names="value")
    delete.on_click(on_delete)

    display(widgets.HBox([picker, show_src, delete]), view)
    render()


show_dashboard()

## Quick test (optional)

Runs the whole pipeline programmatically on a sample paper — a real OpenRouter call, then a save.
Handy for verifying your key/model without using the widgets.

In [ ]:
# Quick test (optional) — real OpenRouter call on a sample paper, then save
SAMPLE_PAPER = (
    "We study the effect of a large-scale conditional cash transfer program on secondary "
    "school enrollment in rural districts. Exploiting the staggered rollout of the program "
    "across 214 districts between 2008 and 2014, we implement a difference-in-differences "
    "design with district and year fixed effects, using administrative enrollment records for "
    "3.1 million students. We find that program exposure increased enrollment by 7.4 percentage "
    "points (SE 1.2), with effects roughly twice as large for girls. Event-study estimates show "
    "no differential pre-trends. Effects are concentrated among households near the eligibility "
    "threshold, and fade after program withdrawal, suggesting limited habit formation. A key "
    "caveat is that migration across district boundaries may bias estimates upward, and "
    "administrative records may undercount informal schooling."
)

result = analyze_paper(SAMPLE_PAPER, DEFAULT_MODEL)
show_brief(result["brief"])
saved_id = save_brief(result["brief"], DEFAULT_MODEL, SAMPLE_PAPER)
print("Saved id:", saved_id)
print("Token/cost usage:", result["usage"])
dashboard_df()